In [ ]:
!pip install git+https://github.com/huggingface/transformers torch accelerate bitsandbytes langchain
!pip install evaluate
!pip install datasets
!pip install git+https://github.com/google-research/bleurt.git
!pip install -U sentence-transformers
!pip install langchain_community
!pip install peft
!pip install rouge_score
!pip install bert_score
!pip install faiss-gpu

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-sgwsrcsb
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-sgwsrcsb
  Resolved https://github.com/huggingface/transformers to commit c1aa0edb48217f416f4bbe6e3a9db1500284513b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 12.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 990.6/990.6 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.8/379.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 

In [ ]:
from datasets import load_metric

def calculate_bleurt(predictions, references):
    bleurt = load_metric("bleurt", config_name="BLEURT-20")
    results = bleurt.compute(predictions=predictions, references=references)
    return results['scores']

In [ ]:
import nltk
from nltk.translate import meteor_score
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

def calculate_meteor(prediction, reference):
    # Tokenize the prediction and reference
    prediction_tokens = word_tokenize(prediction)
    reference_tokens = word_tokenize(reference)
    return meteor_score.meteor_score([reference_tokens], prediction_tokens)

In [ ]:
def evaluate_predictions(predictions, references):
    bleurt_scores = calculate_bleurt(predictions, references)
    meteor_scores = [calculate_meteor(pred, ref) for pred, ref in zip(predictions, references)]

    avg_bleurt = sum(bleurt_scores) / len(bleurt_scores)
    avg_meteor = sum(meteor_scores) / len(meteor_scores)

    return {
        "BLEURT": avg_bleurt,
        "METEOR": avg_meteor,
        "BLEURT_scores": bleurt_scores,
        "METEOR_scores": meteor_scores
    }

In [ ]:
from langchain.vectorstores import FAISS

In [ ]:
!pip install langchain-experimental
from langchain_experimental.text_splitter import SemanticChunker
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/input/coovid/covid_abstracts.csv")
df = df["abstract"]

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings

In [ ]:
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
embedding_model = SentenceTransformerEmbeddings(model_name='BAAI/bge-large-zh-v1.5')

In [ ]:
text_splitter = SemanticChunker(
    embedding_model, breakpoint_threshold_type="percentile"
)
from langchain.docstore.document import Document
docs = [Document(page_content=entry) for entry in df]
docs = text_splitter.split_documents(docs)

In [ ]:
len(df)

In [ ]:
faiss_db = FAISS.from_documents(docs, embedding_model)

In [ ]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

In [ ]:
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from peft import PeftModel
from langchain.retrievers import EnsembleRetriever

In [ ]:
model_name = "filipealmeida/Mistral-7B-Instruct-v0.1-sharded"
base_model = AutoModelForCausalLM.from_pretrained(model_name,
                                             torch_dtype=torch.bfloat16,
                                             quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
!pip install rank_bm25

In [ ]:
from langchain.schema import Document
from langchain_community.retrievers import BM25Retriever
from typing import List

def custom_preprocessing_func(text: str) -> List[str]: # Change the input type to str
    return text.split()

bm25_retriever = BM25Retriever.from_texts(
    [doc.page_content for doc in docs],
    preprocess_func=custom_preprocessing_func
)
bm25_retriever.k = 2

In [ ]:
faiss_retriever = faiss_db.as_retriever(
    search_kwargs={'k': 2}
)

In [ ]:
test = pd.read_csv("/kaggle/input/testtt/testt.csv")

In [ ]:
peft_model_id = "ProElectro07/Projectxx2"
model = PeftModel.from_pretrained(base_model, peft_model_id)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from langchain.chains import SimpleSequentialChain, LLMChain

In [ ]:
##############
text_generation_pipeline = pipeline(
    task="text-generation",
    model=base_model,
    tokenizer=tokenizer,
    temperature=0.3,
    top_k=10,
    top_p=.85,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=500,
    num_return_sequences=1,
    # truncation=False,
    do_sample=True,
    # no_repeat_ngram_size=3,
    # early_stopping=True
)

text_generation_pipeline.model = model

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

In [ ]:

from langchain.retrievers.multi_query import MultiQueryRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.3, 0.7]
)

retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=ensemble_retriever, llm=mistral_llm
)

In [27]:
references = []
for query in test["response"]:
  references.append(query)

In [28]:
from bert_score import score
from evaluate import load
rouge = load('rouge')
bertscore = load('bertscore')
bleu = load('bleu')

In [29]:
###############3
translation_prompt = PromptTemplate(
    template="""
<s>[INST]
You are an experienced linguistic expert and an assitant to a doctor who only understand english.
You will be given code-mixed hindi-english queries and you need to translate them into english.

Here is an example of a good response:
Patient: Doctor sahib, mumjhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Reponse: Doctor, I have fever and cough, are these things symptoms of covid?

Now, please translate the patient's query below in a similar manner, in English:
Patient's question: [{input}]
Provide an accurate translated query which could :
[/INST] </s>
"""
)

translation_chain = LLMChain(llm=mistral_llm, prompt=translation_prompt)

/opt/conda/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  warn_deprecated(


In [30]:
#########################
prompt_template = """<s>[INST]
You are an experienced COVID-19 doctor, who will help out the patient with their queries.

Here is an example of a good response:
Patient: Doctor, I have fever and cough, are these things symptoms of covid?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient's question: {question}

Additional context that might help the patient's query: {context}

Provide a clear and concise response to the patient's:
[/INST] </s>
"""

# Create a prompt instance
Prompt_Template = PromptTemplate.from_template(prompt_template)

# Create the RAG chain
RAG_chain = RetrievalQA.from_chain_type(
    mistral_llm,
    retriever=retriever_from_llm,
    chain_type_kwargs={"prompt": Prompt_Template}
)
combined_chain = SimpleSequentialChain(
    chains=[translation_chain, RAG_chain],
    verbose=True
)

predictions = []
count = 0
for query in test["queries"]:
    response = combined_chain({"input": query})
    # count = count + 1
    # response = RAG_chain({"query": query["text"]})
    predictions.append(response["output"])
    print(response["output"])

/opt/conda/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(




> Entering new SimpleSequentialChain chain...
Patient: Good day. This morning, for the first time, I felt a cough. I was told that I needed to get a report done due to the coronavirus. I coughed for 5 minutes. I don't have a fever or any pain in my chest. What should I do?

Thank you for reaching out. Based on your symptoms, it is possible that you may have contracted COVID-19. However, it is important to note that not everyone with COVID-19 experiences a cough, so it is also possible that you may not have the virus.

In either case, it is important to take precautions to protect yourself and others. If you believe you may have been exposed to COVID-19, you should self-isolate and monitor your symptoms closely. If you develop a fever or difficulty breathing, seek medical attention immediately.

It is also recommended that you get tested for COVID-19 as soon as possible. Testing can confirm whether or not you have the virus and provide guidance on how to proceed.

In the meantime, ple

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Patient: My child has been sick for two weeks now. He hasn't spoken yet, so it's difficult to determine if he has any difficulty breathing or not. Is my child's cough a problem?

Doctor: A persistent cough in a child, especially one that lasts for more than two weeks, could be a sign of COVID-19. However, it's important to note that children with COVID-19 may not always show typical symptoms such as fever or difficulty breathing. If your child is showing signs of distress, such as rapid breathing, lethargy, or difficulty feeding, seek immediate medical attention. In the meantime, monitor your child's symptoms closely and encourage them to rest and stay hydrated. It's also important to keep your child away from others and follow all necessary precautions to prevent the spread of COVID-19.

> Finished chain.

Patient: My child has been sick for two weeks now. He hasn't spoken yet, so it's difficult to determine if he has any difficulty breathing or not. Is my child's cough a problem?

D

In [33]:
predictions

['\nThank you for reaching out. Based on your symptoms, it is possible that you may have contracted COVID-19. However, it is important to note that not everyone with COVID-19 experiences a cough, so it is also possible that you may not have the virus.\n\nIn either case, it is important to take precautions to protect yourself and others. If you believe you may have been exposed to COVID-19, you should self-isolate and monitor your symptoms closely. If you develop a fever or difficulty breathing, seek medical attention immediately.\n\nIt is also recommended that you get tested for COVID-19 as soon as possible. Testing can confirm whether or not you have the virus and provide guidance on how to proceed.\n\nIn the meantime, please continue to practice good hygiene, such as washing your hands frequently and wearing a mask when in public spaces. Additionally, avoid close contact with others who are sick and stay home if you feel unwell.\n\nI hope this information helps. Please let me know if

In [31]:
predictions_fixed = [p for p in predictions]
references_fixed = [r for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 3.76 seconds, 13.29 sentences/sec
ROUGE scores: {'rouge1': 0.19891855272537695, 'rouge2': 0.02151166699717043, 'rougeL': 0.11272413225862196, 'rougeLsum': 0.12269053070269029}
BERT scores: {'precision': 0.8241027593612671, 'recall': 0.8273252844810486, 'f1': 0.8256242871284485}
BLEU score: {'bleu': 0.004031900919398234, 'precisions': [0.1556188824185355, 0.010721855130283007, 0.0012545477355413374, 0.00012624668602449185], 'brevity_penalty': 1.0, 'length_ratio': 2.359251680795089, 'translation_length': 8071, 'reference_length': 3421}


In [32]:
from langchain.retrievers.multi_query import MultiQueryRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[1, 0]
)

retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=ensemble_retriever, llm=mistral_llm
)

In [34]:
# Create the RAG chain
RAG_chain = RetrievalQA.from_chain_type(
    mistral_llm,
    retriever=retriever_from_llm,
    chain_type_kwargs={"prompt": Prompt_Template}
)
combined_chain = SimpleSequentialChain(
    chains=[translation_chain, RAG_chain],
    verbose=True
)

predictions = []
count = 0
for query in test["queries"]:
    response = combined_chain({"input": query})
    predictions.append(response["output"])
    print(response["output"])



> Entering new SimpleSequentialChain chain...
Patient: Good day. This morning, I had my first cough after a long time. It was reported that I have been infected with corona virus. I coughed for 5 minutes. I don't have fever or chest pain. What should I do now?

Hello! Thank you for reaching out. Based on your symptoms of a persistent cough, it is important to take this seriously and follow up with your healthcare provider. While a single coughing episode may not necessarily indicate COVID-19, it is still possible that you could have been exposed to the virus. Your healthcare provider can assess your symptoms and determine whether testing for COVID-19 is necessary. In the meantime, it is important to practice self-care by getting plenty of rest, staying hydrated, and avoiding close contact with others. If your symptoms worsen or you experience any new symptoms such as fever, shortness of breath, or chest pain, please seek medical attention immediately. Remember, COVID-19 can be seriou

In [35]:
predictions_fixed = [p for p in predictions]
references_fixed = [r for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 5.26 seconds, 9.51 sentences/sec
ROUGE scores: {'rouge1': 0.18308046289069585, 'rouge2': 0.0200943923596356, 'rougeL': 0.10334594568437652, 'rougeLsum': 0.11682381728992972}
BERT scores: {'precision': 0.8205739855766296, 'recall': 0.8243871331214905, 'f1': 0.8223800659179688}
BLEU score: {'bleu': 0.0041468986960207345, 'precisions': [0.14099977583501458, 0.010144274120829576, 0.001813647698934482, 0.00011399908800729594], 'brevity_penalty': 1.0, 'length_ratio': 2.6080093539900613, 'translation_length': 8922, 'reference_length': 3421}


In [ ]:
#########################
prompt_template = """<s>[INST]
You are an experienced COVID-19 doctor, who will help out the patient with their queries.

Here is an example of a good response:
Patient: Doctor, I have fever and cough, are these things symptoms of covid?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient's question: {question}

Additional context that might help the patient's query: {context}

Provide a clear and concise response to the patient's:
[/INST] </s>
"""

# Create a prompt instance
Prompt_Template = PromptTemplate.from_template(prompt_template)

# Create the RAG chain
check = RetrievalQA.from_chain_type(
    mistral_llm,
    retriever=retriever_from_llm,
    chain_type_kwargs={"prompt": Prompt_Template}
)
combined_chain = SimpleSequentialChain(
    chains=[translation_chain, RAG_chain],
    verbose=True
)

predictions = []
count = 0
for query in test["queries"]:
    response = combined_chain({"input": query})
    # count = count + 1
    # response = RAG_chain({"query": query["text"]})
    predictions.append(response["output"])
    print(response["output"])